[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/eight_conics/eight_conics.ipynb)

# Penrose's Eight-Conic Theorem

This example computes Penrose's eight-conic theorem for quadrics touching in pairs along the edges of a cube. Two quadrics are in double contact when they meet at two points and share their tangent planes at both crossings. When seven quadrics occupy seven corners of a cube so that adjacent quadrics stand in double contact and the four contact planes around each face meet in a common point, an eighth quadric completes the cube in double contact with all three remaining neighbours, following the construction in [Arnold, Chern, Eide, Gunn, Neukirchner, and Penrose (2024)](https://arxiv.org/html/2409.17150v8#S6).

Each quadric is modelled as a symmetric polarity map sending points to polar planes, so `quadric(point) & point` vanishes on the locus and `quadric(point)` gives the tangent plane at every point of the curve. Adding the plane dyad `contact_plane * (contact_plane & Point)` vanishes wherever `contact_plane & point == 0`, so `base + contact_plane * (contact_plane & Point)` shares both crossings and both tangent planes with `base` along `contact_plane`. Around the missing cube corner, adding each closing plane's dyad `closing_planes * (closing_planes & Point)` to its scaled face quadric gives three algebraic completions that evaluate to the same polarity map.

In homogeneous matrix notation, homogeneous point column vectors $\mathbf{x} = [x, y, w]^{\mathsf{T}}$ and line coordinate vectors $\boldsymbol{\ell} = [\ell_x, \ell_y, \ell_w]^{\mathsf{T}}$ write point conics as quadratic forms $\mathbf{x}^{\mathsf{T}} C \mathbf{x} = 0$ on symmetric $3 \times 3$ matrices $C$, dual line envelopes as $\boldsymbol{\ell}^{\mathsf{T}} C^{-1} \boldsymbol{\ell} = 0$, and line concurrence as $3 \times 3$ determinants $\det[\boldsymbol{\ell}_a, \boldsymbol{\ell}_b, \boldsymbol{\ell}_c] = 0$. Double contact along $\boldsymbol{\ell}^{\mathsf{T}} \mathbf{x} = 0$ takes the outer-product pencil form $C' = C + \lambda\,\boldsymbol{\ell}\boldsymbol{\ell}^{\mathsf{T}}$, and coupling pairs of lines on each cube face forms $(1 - \mu_{ij}^2) C + \boldsymbol{\ell}_i\boldsymbol{\ell}_i^{\mathsf{T}} + \boldsymbol{\ell}_j\boldsymbol{\ell}_j^{\mathsf{T}} + \mu_{ij}(\boldsymbol{\ell}_i\boldsymbol{\ell}_j^{\mathsf{T}} + \boldsymbol{\ell}_j\boldsymbol{\ell}_i^{\mathsf{T}})$ before closing the eighth conic with three matching rank-one outer products.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga
from examples.math.eight_conics import render
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                                              # Vector+
Point = ga.gatype.antivector()                                          # Antivector-
Quadric = ga.gatype((Plane, Point))                                     # Plane+ <- Point-

first_axes = np.array([1, 2, 0])
second_axes = np.array([2, 0, 1])
edges = np.array([
    [0, 1], [0, 2], [0, 4], [1, 3], [2, 3], [1, 5],
    [4, 5], [2, 6], [4, 6], [3, 7], [5, 7], [6, 7],
])

## Double contact along a plane

Adding a scaled plane dyad `strength * plane * (plane & Point)` to a base quadric leaves its polar planes unchanged wherever `plane & point` vanishes. Both quadrics therefore cross that plane at the same two points and share their tangent planes at both crossings regardless of `strength`.

In [ ]:
strength = 2.5
pair_offset = 0.5
pair_weight = 0.7

center = mv.w.dual()                                                    # [] Point-
semi_axes = stack((mv.x.dual(), mv.y.dual()))                           # [2] Point-
pair_plane = (mv.x - mv.w * pair_offset) * pair_weight                  # [] Plane+

In [ ]:
def ellipsoid(center: Point, semi_axes: Point) -> Quadric:
    """The dual ellipsoid on `center` (`Point-`) and `semi_axes` (`[2] Point-`) inverts to its point-to-plane polarity map (`Plane+ <- Point-`)."""
    dual_quadric = (semi_axes * (Plane & semi_axes)).sum(axis=0) - center * (Plane & center)  # [] Point- <- Plane+
    return dual_quadric.inverse()                                                             # [] Plane+ <- Point-


def intersections(quadrics: Quadric, planes: Plane) -> Point:
    """Each quadric meets its plane in two real points (`[..., 2] Point-`) along the plane's tangent direction."""
    origin = mv.w.dual()                                                    # [] Point-
    unit_planes = planes.normalized()                                       # [...] Plane+
    # Averaging the chart origin with its reflection lands on the perpendicular foot on each plane.
    anchors = (origin + (unit_planes >> origin)) / 2                        # [...] Point-
    directions = (unit_planes | mv.xy).dual()                               # [...] Point-
    quadratic_terms = quadrics(directions) & directions                     # [...] Scalar+
    linear_terms = quadrics(anchors) & directions                           # [...] Scalar+
    constant_terms = quadrics(anchors) & anchors                            # [...] Scalar+
    root_signs = np.array([-1, 1])
    discriminants = linear_terms.squared() - quadratic_terms * constant_terms  # [...] Scalar+
    offsets = (-linear_terms[..., None] + discriminants.square_root()[..., None] * root_signs) / quadratic_terms[..., None]  # [..., 2] Scalar+
    return anchors[..., None] + directions[..., None] * offsets             # [..., 2] Point-


sphere = ellipsoid(center, semi_axes)                                   # [] Plane+ <- Point-
# The plane dyad vanishes on `pair_plane`, preserving both crossings and both tangent planes there.
double_plane = pair_plane * (pair_plane & Point)                        # [] Plane+ <- Point-
contact_quadric = sphere + strength * double_plane                      # [] Plane+ <- Point-
pair_contacts = intersections(sphere, pair_plane)                       # [2] Point-
save_figure(render.double_contact_figure(sphere, contact_quadric, pair_contacts), "eight_conics_double_contact");

The blue circle and the orange ellipse share both marked crossings along the dashed contact plane and touch the same tangent line at each crossing.

## Closing the cube of eight quadrics

Placing a base quadric at one corner of a cube and adding three contact dyads gives three neighbour quadrics along the incident edges. Coupling each pair of those contact planes with a symmetric cross term produces three face quadrics whose four edge contact planes concur on every face. Adding one closing contact dyad to each scaled face quadric then yields three independent formulas for the eighth corner that evaluate to a single polarity map throughout a continuous deformation. Rendering the animation across the loop takes about fifteen seconds.

In [ ]:
plane_angles = np.arange(3) * (2 * np.pi / 3)
plane_offsets = np.array([0.20, 0.24, 0.16])
cube_plane_weight = 1.3
mean_couplings = np.array([0.30, -0.24, 0.27])
coupling_amplitude = 0.25
n_frames = 216
frame_duration_ms = 40
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
couplings = mean_couplings + coupling_amplitude * np.cos(phases[:, None] + plane_angles)
closing_edges = np.array([9, 10, 11])

plane_rotors = (mv.xy * (-plane_angles / 2)).exp()                      # [3] Rotor+
normals = plane_rotors >> mv.x                                          # [3] Plane+
contact_planes = (normals - mv.w * plane_offsets) * cube_plane_weight   # [3] Plane+

In [ ]:
def configuration(base: Quadric, contact_planes: Plane, couplings: np.ndarray) -> tuple[Quadric, Plane, Quadric]:
    """A base quadric, three contact planes (`[3] Plane+`), and three face couplings (`[..., 3]`) determine the cube's eight quadrics (`[..., 8]`), twelve edge contact planes (`[..., 12]`), and three eighth-vertex completions (`[..., 3]`)."""
    double_planes = contact_planes * (contact_planes & Point)               # [3] Plane+ <- Point-
    neighbours = base + double_planes                                       # [3] Plane+ <- Point-
    first_planes = contact_planes[first_axes]                               # [3] Plane+
    second_planes = contact_planes[second_axes]                             # [3] Plane+
    face_scales = 1 - couplings ** 2

    # Each face quadric shares one contact plane with each of two neighbours.
    cross_terms = first_planes * (second_planes & Point) + second_planes * (first_planes & Point)  # [3] Plane+ <- Point-
    faces = (
        base * face_scales
        + double_planes[first_axes]
        + double_planes[second_axes]
        + cross_terms * couplings
    )                                                                       # [..., 3] Plane+ <- Point-

    # The three closing contact planes determine the same eighth quadric independently.
    closing_scale = 1 - np.sum(couplings ** 2, axis=-1) - 2 * np.prod(couplings, axis=-1)
    closing_planes = (
        contact_planes * face_scales
        + first_planes * (couplings[..., second_axes] + couplings * couplings[..., first_axes])
        + second_planes * (couplings[..., first_axes] + couplings * couplings[..., second_axes])
    )                                                                       # [..., 3] Plane+
    completions = (
        faces * closing_scale[..., None]
        + closing_planes * (closing_planes & Point)
    ) / face_scales                                                         # [..., 3] Plane+ <- Point-

    # Binary vertex labels flip one of the three choices along each cube edge.
    batch_shape = completions.shape[:-1]
    quadrics = stack((
        base.broadcast_to(batch_shape),
        neighbours[0].broadcast_to(batch_shape),
        neighbours[1].broadcast_to(batch_shape),
        faces[..., 2],
        neighbours[2].broadcast_to(batch_shape),
        faces[..., 1],
        faces[..., 0],
        completions[..., 0],
    ), axis=-1)                                                             # [..., 8] Plane+ <- Point-
    first_edge_planes = first_planes * couplings + second_planes            # [..., 3] Plane+
    second_edge_planes = first_planes + second_planes * couplings           # [..., 3] Plane+
    edge_planes = stack((
        contact_planes[0].broadcast_to(batch_shape),
        contact_planes[1].broadcast_to(batch_shape),
        contact_planes[2].broadcast_to(batch_shape),
        first_edge_planes[..., 2],
        second_edge_planes[..., 2],
        second_edge_planes[..., 1],
        first_edge_planes[..., 1],
        first_edge_planes[..., 0],
        second_edge_planes[..., 0],
        closing_planes[..., 2],
        closing_planes[..., 1],
        closing_planes[..., 0],
    ), axis=-1)                                                             # [..., 12] Plane+
    return quadrics, edge_planes, completions


def deformation(
    base: Quadric,
    contact_planes: Plane,
    couplings: np.ndarray,
) -> Iterator[tuple[Quadric, Plane, Point, Quadric]]:
    """Yields `(quadrics, edge_planes, contacts, completions)` (`[8]`, `[12]`, `[12, 2]`, `[3]`) as the face couplings (`[n_frames, 3]`) deform the cube."""
    for coupling in couplings:
        quadrics, edge_planes, completions = configuration(base, contact_planes, coupling)
        contacts = intersections(quadrics[edges[:, 0]], edge_planes)        # [12, 2] Point-
        yield quadrics, edge_planes, contacts, completions


cube_states = show_states(
    deformation(sphere, contact_planes, couplings),
    render.eighth_conic_animation,
    edges,
    closing_edges,
    name="eight_conics_eighth",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
)

The bold orange curve shows the eighth quadric, while the blue, green, and pink curves show its three cube neighbours together with their dashed closing contact planes and six marked points of tangency, over the four remaining quadrics in pale grey. As the three face couplings sweep around a closed loop, all eight quadrics deform continuously while the eighth quadric maintains double contact with all three neighbours.

In [ ]:
first_quadrics = cube_states[0][0]
print(first_quadrics[-1].coefficient_table())

In [ ]:
# checks
assert ((sphere(pair_contacts) & pair_contacts).abs() < 1e-9).all()
assert ((contact_quadric(pair_contacts) & pair_contacts).abs() < 1e-9).all()
assert (np.abs((contact_quadric(pair_contacts) - sphere(pair_contacts)).kernel) < 1e-9).all()

streamed_quadrics = stack([state[0] for state in cube_states])              # [n_frames, 8] Plane+ <- Point-
streamed_contacts = stack([state[2] for state in cube_states])              # [n_frames, 12, 2] Point-
streamed_completions = stack([state[3] for state in cube_states])           # [n_frames, 3] Plane+ <- Point-
assert (np.abs((streamed_completions[:, 1:] - streamed_completions[:, :1]).kernel) < 1e-9).all()

for end in range(2):
    edge_quadrics = streamed_quadrics[:, edges[:, end]][..., None]          # [n_frames, 12, 1] Plane+ <- Point-
    assert ((edge_quadrics(streamed_contacts) & streamed_contacts).abs() < 1e-9).all()